<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session03-databases-in-code.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 3 — Biological Databases in code {.unnumbered}

## Goal

Use supplied identifiers to retrieve a few small database records, parse their fields and check what the results represent. First compare one shared archival sequence through two interfaces. Then inspect one *HBB* transcript and a linked UniProt JSON record. The optional final section uses a literature search API.

This is a demonstration of **programmatic access**, rather than a completed database dossier. The book introduces the biological levels and resource types. In the lab, you choose records, follow their links and document the evidence yourself. Read the saved outputs without running code, or work through the cells at your own pace. Rerunning needs an internet connection; dates and annotations can change.

## Setup

In [1]:
# Install only if needed. No installation is required to read the saved outputs.
import importlib.util
import subprocess
import sys

missing = [package for package in ("biopython", "requests")
           if importlib.util.find_spec("Bio" if package == "biopython" else package) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [2]:
import io
from datetime import datetime, timezone
import requests
from Bio import SeqIO

print("Retrieved (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Python:", sys.version.split()[0])

Retrieved (UTC): 2026-10-30T12:29:48+00:00
Python: 3.9.13


## One submitted record, two partner interfaces {#insdc-two-interfaces}

The submitted INSDC record `V00497.1` is **human beta-globin (*HBB*) mRNA**, not a whole genomic region. Open it in the [NCBI nucleotide interface](https://www.ncbi.nlm.nih.gov/nuccore/V00497.1) and the [ENA browser](https://www.ebi.ac.uk/ena/browser/view/V00497.1). The pages and file formats look different. Do they describe the same archived sequence?

The next cell retrieves this one small record through NCBI's API in GenBank format and ENA's API in EMBL format. It checks the full accession and sequence version, organism, length and sequence, rather than comparing only the title. No complete database download is needed.

In [3]:
import io
from datetime import datetime, timezone

archive_accession = "V00497.1"
archive_sources = {
    "NCBI": (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        "?db=nuccore&id=V00497.1&rettype=gb&retmode=text",
        "genbank",
    ),
    "ENA": (
        "https://www.ebi.ac.uk/ena/browser/api/embl/V00497.1?download=false",
        "embl",
    ),
}
archive_records = {}
for partner, (url, file_format) in archive_sources.items():
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    archive_records[partner] = SeqIO.read(io.StringIO(response.text), file_format)

def archive_taxon(record):
    return next(
        xref.split(":", 1)[1]
        for feature in record.features if feature.type == "source"
        for xref in feature.qualifiers.get("db_xref", [])
        if xref.startswith("taxon:")
    )

print("Retrieved (UTC):", datetime.now(timezone.utc).date())
print(f"{'Partner':8s} {'Accession.version':18s} {'Organism label':22s} {'Taxon':6s} Length (bases)")
for partner, archive_record in archive_records.items():
    print(f"{partner:8s} {archive_record.id:18s} "
          f"{archive_record.annotations['organism']:22s} {archive_taxon(archive_record):6s} {len(archive_record.seq)}")

ncbi_archive = archive_records["NCBI"]
ena_archive = archive_records["ENA"]
archive_checks = {
    "same accession.version": ncbi_archive.id == ena_archive.id == archive_accession,
    "same organism (taxonomy ID)": archive_taxon(ncbi_archive) == archive_taxon(ena_archive) == "9606",
    "same length": len(ncbi_archive.seq) == len(ena_archive.seq),
    "same sequence": str(ncbi_archive.seq).upper() == str(ena_archive.seq).upper(),
}
for check, passed in archive_checks.items():
    print(f"{check}: {passed}")
assert all(archive_checks.values()), "Inspect the two returned records before continuing."

Retrieved (UTC): 2026-10-30
Partner  Accession.version  Organism label         Taxon  Length (bases)
NCBI     V00497.1           Homo sapiens           9606   626
ENA      V00497.1           Homo sapiens (human)   9606   626
same accession.version: True
same organism (taxonomy ID): True
same length: True
same sequence: True


The organism labels have different formatting: `Homo sapiens` at NCBI and `Homo sapiens (human)` at ENA. Both source features identify taxon `9606`. Comparing that identifier checks the organism without mistaking a common name in parentheses for a biological difference.

**Check the evidence.** These are two access routes to the **same submitted record**. Agreement checks retrieval and identity; it is not two independent experiments supporting the biological annotation. Read the record's source, features, references and submission history before drawing biological conclusions. If a request fails, keep the accession and try another partner; do not interpret a failed link as evidence that the data never existed.

`V00497.1` is an original submitted sequence. The `NM_000518.5` RefSeq mRNA used below is a separate reference record. INSDC partners exchange submitted archival records; this does not make every NCBI resource, including RefSeq or Gene, an INSDC mirror. Nor should these two *HBB* mRNA accessions be assumed to have identical sequences without comparing them.

## Linked records describe different parts of an investigation {#insdc-record-roles}

| Record role | What to check |
|---|---|
| Study / BioProject | Project description, purpose and links to its deposited data. |
| Sample / BioSample | Biological source, organism and recorded collection or treatment information. |
| Experiment | How the sequencing library was prepared and which instrument was used. |
| Reads / run | Sequence reads and their quality scores from a sequencing run. |
| Assembly / assembled sequence | A submitted reconstructed sequence, its assembly context and available annotation. |

These roles link the **context, source and data** of an investigation. They are different from the DNA, RNA and protein levels followed in the next section. Some investigations have several samples and runs, and older sequence records may not link to every role. Follow the links actually present in a record; do not invent a project or read set for `V00497.1`. An archive stores a submitted assembly; receiving raw reads does not automatically assemble them.

**Optional: follow a changing archive route.** Inspect the metadata for run [`SRR9451988` at ENA](https://www.ebi.ac.uk/ena/browser/view/SRR9451988) and its experiment [`SRX6217698` at NCBI SRA](https://www.ncbi.nlm.nih.gov/sra/SRX6217698). Its NISC trace-conversion history connects it to the older capillary Trace Archives discussed in the lecture. Find the organism, library label and sequencing technology. This example is *Papio anubis* (olive baboon), **not human**. The library label `HBB` alone does not demonstrate that every read covers the *HBB* gene; establishing that would require sequence comparison. Read sequences and quality scores in a current read archive do not by themselves establish that original chromatogram files are available through every interface. Inspecting metadata is sufficient here; do not download the full read set. The general habit is to retain the accession and follow the current archive route when an old website changes.

## Read one supplied transcript record {#hbb-transcript-checks}

The next identifier is supplied so we can concentrate on retrieval and interpretation: RefSeq transcript `NM_000518.5`, human *HBB* mRNA. This is a separate reference record from the submitted INSDC sequence compared above. We request one GenBank-format record, check the HTTP response and parse it with Biopython. The URL is printed so the request can be repeated outside Python.

In [4]:
transcript_accession = "NM_000518.5"
transcript_response = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
    params={"db": "nuccore", "id": transcript_accession, "rettype": "gb", "retmode": "text"},
    timeout=30,
)
transcript_response.raise_for_status()
mrna = SeqIO.read(io.StringIO(transcript_response.text), "genbank")

print("Source:", transcript_response.url)
print("Accession.version:", mrna.id)
print("Description:", mrna.description)
print("Organism:", mrna.annotations["organism"])
print("Recorded molecule type:", mrna.annotations["molecule_type"])
print("Transcript length:", len(mrna.seq), "bases")
print("First 60 bases:", mrna.seq[:60])
assert mrna.id == transcript_accession
assert mrna.annotations["organism"] == "Homo sapiens"
assert mrna.annotations["molecule_type"] == "mRNA"

Source: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NM_000518.5&rettype=gb&retmode=text
Accession.version: NM_000518.5
Description: Homo sapiens hemoglobin subunit beta (HBB), mRNA
Organism: Homo sapiens
Recorded molecule type: mRNA
Transcript length: 628 bases
First 60 bases: ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACCATGGTGCATC


### Inspect the annotation, including its links

The record describes mRNA although its sequence is written with **T**. DNA notation alone does not tell us whether a record represents genomic DNA, RNA or a physical cDNA experiment. Read the molecule type and annotation.

The `gene` feature links this transcript to a gene identifier. The `CDS` feature identifies its protein-coding region and linked protein accession. These are different kinds of identifier, even when one record includes them together. GenBank displays positions starting at 1 with both ends included. Biopython uses Python slice coordinates starting at 0 with the end excluded, so the code converts the start before printing.

In [5]:
gene_feature = next(feature for feature in mrna.features if feature.type == "gene")
cds_feature = next(feature for feature in mrna.features if feature.type == "CDS")
gene_links = [link for link in gene_feature.qualifiers.get("db_xref", [])
              if link.startswith("GeneID:")]
protein_accession = cds_feature.qualifiers["protein_id"][0]

print("Gene symbol:", gene_feature.qualifiers["gene"][0])
print("Linked gene identifier(s):", ", ".join(gene_links))
print("Linked protein accession:", protein_accession)
print("CDS on this transcript (1-based, inclusive):",
      int(cds_feature.location.start) + 1, "to", int(cds_feature.location.end))
assert "GeneID:3043" in gene_links
assert protein_accession == "NP_000509.1"

Gene symbol: HBB
Linked gene identifier(s): GeneID:3043
Linked protein accession: NP_000509.1
CDS on this transcript (1-based, inclusive): 51 to 494


### Check what the coding region contains

A full mature mRNA includes untranslated regions (UTRs) as well as its CDS. Extract the CDS using its annotated location instead of translating the whole transcript or guessing where translation begins. For this complete, standard-code example, `cds=True` also checks the start codon, length and terminal stop codon. The stop codon is part of the CDS nucleotide span but does not add an amino-acid residue.

In [6]:
cds_sequence = cds_feature.extract(mrna.seq)
translation_table = int(cds_feature.qualifiers.get("transl_table", ["1"])[0])
translated_cds = cds_sequence.translate(table=translation_table, cds=True)
annotated_translation = cds_feature.qualifiers["translation"][0]

print("Full transcript:", len(mrna.seq), "bases")
print("CDS including terminal stop:", len(cds_sequence), "bases")
print("Bases outside the annotated CDS:", len(mrna.seq) - len(cds_sequence))
print("Encoded protein:", len(translated_cds), "residues")
print("Translated CDS matches the record's translation:",
      str(translated_cds) == annotated_translation)
assert str(translated_cds) == annotated_translation
assert len(cds_sequence) == 3 * (len(translated_cds) + 1)

Full transcript: 628 bases
CDS including terminal stop: 444 bases
Bases outside the annotated CDS: 184
Encoded protein: 147 residues
Translated CDS matches the record's translation: True


**Interpret the checks.** They confirm that we retrieved the requested record and interpreted its annotated coding region consistently. Comparing a computed translation with the same record's annotation is a consistency check, not independent experimental confirmation.

**Two transcripts, same gene?** If a second transcript record links to `GeneID:3043` in the same annotation context, that links it to the same annotated human gene. Distinct transcript accessions identify different records; sequence versions, source records and transcript variants need to be distinguished. The comparison above does not demonstrate alternative splicing in *HBB*. The lecture's three-exon diagram illustrates possible mechanisms generally. In the lab, inspect the links and annotations of your selected records before deciding how they relate.

## Parse one linked protein annotation {#uniprot-record-checks}

UniProt provides structured JSON as well as web pages. Here the supplied accession is `P68871`. Request only this entry and inspect selected fields. A shared gene name is insufficient to establish a particular transcript–protein relationship, so we also check its explicit RefSeq cross-references.

In [7]:
uniprot_response = requests.get("https://rest.uniprot.org/uniprotkb/P68871.json", timeout=30)
uniprot_response.raise_for_status()
entry = uniprot_response.json()
refseq_links = [link for link in entry.get("uniProtKBCrossReferences", [])
               if link["database"] == "RefSeq"]
linked_proteins = {link["id"] for link in refseq_links}
linked_transcripts = {prop["value"] for link in refseq_links
                      for prop in link.get("properties", [])
                      if prop["key"] == "NucleotideSequenceId"}

print("Source:", uniprot_response.url)
print("UniProt accession:", entry["primaryAccession"])
print("Entry type:", entry["entryType"])
print("Organism:", entry["organism"]["scientificName"])
print("Protein existence:", entry["proteinExistence"])
print("Our protein is explicitly cross-referenced:", protein_accession in linked_proteins)
print("Our transcript is explicitly cross-referenced:", transcript_accession in linked_transcripts)
assert entry["primaryAccession"] == "P68871"
assert entry["organism"]["taxonId"] == 9606
assert protein_accession in linked_proteins
assert transcript_accession in linked_transcripts

Source: https://rest.uniprot.org/uniprotkb/P68871.json
UniProt accession: P68871
Entry type: UniProtKB reviewed (Swiss-Prot)
Organism: Homo sapiens
Protein existence: 1: Evidence at protein level
Our protein is explicitly cross-referenced: True
Our transcript is explicitly cross-referenced: True


### Read evidence attached to a particular claim

Reviewed status describes curation of an entry. The protein-existence field concerns evidence that the protein exists. Neither substitutes for the evidence attached to an individual functional claim. The next cell prints one function statement and its own evidence links. Read the linked source before deciding whether it supports the conclusion you want to make.

In [8]:
function_statement = next(text for comment in entry.get("comments", [])
                          if comment["commentType"] == "FUNCTION"
                          for text in comment.get("texts", []) if text.get("evidences"))
print("Function statement:", function_statement["value"])
for evidence in function_statement["evidences"]:
    print("Evidence:", evidence["evidenceCode"], evidence.get("source", ""), evidence.get("id", ""))
    if evidence.get("source") == "PubMed":
        print("Reference:", "https://pubmed.ncbi.nlm.nih.gov/" + evidence["id"] + "/")

Function statement: Involved in oxygen transport from the lung to the various peripheral tissues
Evidence: ECO:0000269 PubMed 28066926
Reference: https://pubmed.ncbi.nlm.nih.gov/28066926/


## Optional: check a literature search through an API {#pubmed-query-checks}

Later in the lecture we use online literature searches. This example retrieves counts for two supplied terms and their Boolean combinations. It demonstrates query parameters, JSON parsing and a reasonableness check, rather than a complete literature review. Requests are spaced to respect NCBI's unauthenticated rate limit and retried if the server asks us to slow down.

In [9]:
import time

def pubmed_count(term):
    for attempt in range(5):
        time.sleep(0.5)
        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={"db": "pubmed", "term": term, "retmode": "json"}, timeout=30,
        )
        if response.status_code != 429:
            break
        time.sleep(2 * (attempt + 1))
    response.raise_for_status()
    return int(response.json()["esearchresult"]["count"])

a = pubmed_count("hemoglobin")
b = pubmed_count("malaria")
a_and_b = pubmed_count("hemoglobin AND malaria")
a_or_b = pubmed_count("hemoglobin OR malaria")
a_not_b = pubmed_count("hemoglobin NOT malaria")

print(f"hemoglobin              {a:>10,}")
print(f"malaria                 {b:>10,}")
print(f"hemoglobin AND malaria  {a_and_b:>10,}")
print(f"hemoglobin OR malaria   {a_or_b:>10,}   (predicted: {a + b - a_and_b:,})")
print(f"hemoglobin NOT malaria  {a_not_b:>10,}   (predicted: {a - a_and_b:,})")
print("OR count matches set arithmetic:", a_or_b == a + b - a_and_b)
print("NOT count matches set arithmetic:", a_not_b == a - a_and_b)

hemoglobin                 289,650
malaria                    125,734
hemoglobin AND malaria       4,470
hemoglobin OR malaria      410,914   (predicted: 410,914)
hemoglobin NOT malaria     285,180   (predicted: 285,180)
OR count matches set arithmetic: True
NOT count matches set arithmetic: True


## Checks and next steps

The comparison of NCBI and ENA tested the identity of one shared submitted record. The transcript cells checked the requested molecule type, linked gene and protein, coding-region coordinates and translation. The JSON cells checked explicit links and displayed evidence for one claim. These checks illustrate how code can support careful reading of a record.

The literature counts should follow set union and set difference when the searches describe the same sets. If a live result differs, inspect PubMed's query translation and repeat the requests; indexing can change between calls. Search counts change over time and do not measure the quality of the papers.

In the lab, apply this approach to records you select: document the biological object, source URL, accession and sequence version where applicable, retrieval date, actual cross-reference and relevant evidence. The book supplies the conceptual framework for genomic context, transcript diversity, noncoding RNA, expression and protein information. A complete database dossier belongs to that lab investigation, rather than to this small API demonstration.